# Unitary completions in the Weyl chamber

A physical two-rail gate that preserves the dual-rail code space
$\mathcal{C} = \operatorname{span}\{|01\rangle, |10\rangle\}$ has the block form
$M(A, U_L)$: the logical gate $U_L$ on $\mathcal{C}$ and a free complement
$A \in \mathrm{U}(2)$ on $\{|00\rangle, |11\rangle\}$. Three tiers, each computed
analytically from the input $U_L$: all physical completions (blue: exact planar
faces, fitted from deterministic Haar samples of $A$), the loss-compatible
admissible family $A = e^{i\gamma} R_z(\beta)$ (orange: the Weyl fold of the line
$c_1 = c_2 = \phi_B$ with $\cos\phi_B = |(U_L)_{11}|$), and the realizations the encoding actually retains, $U(C_r)$ for each
circuit $r \in \mathcal{R}(g)$, composed from the lowered realization sets
(red points; for diagonal gates these are rail-local and sit at the origin). The chamber shows local-equivalence
classes; edit `TARGETS` to plot other logical gates.

In [ ]:
from fractions import Fraction
from itertools import combinations, product
from pathlib import Path

import lusca  # noqa: F401  registers the "lusca" matplotlib style
import matplotlib.pyplot as plt
import numpy as np
from gulps import GateInvariants
from gulps.viz.weyl_chamber import WeylChamber
from mpl_toolkits.mplot3d.art3d import Line3DCollection, Poly3DCollection
from qiskit import QuantumCircuit
from qiskit.circuit.library import XGate
from qiskit.quantum_info import Operator
from qiskit.quantum_info.random import random_unitary
from scipy.linalg import expm
from scipy.spatial import ConvexHull

from marabbecca.encodings import DualRail

TARGETS = [
    XGate(),
    XGate().power(0.5),
]  # logical unitaries, one panel each (the paper figure)

ROOT = next(
    p
    for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if (p / "pyproject.toml").exists()
)
FIG_DIR = ROOT / "build/figures"

In [ ]:
CHAMBER = np.array([(0, 0, 0), (1, 0, 0), (0.5, 0.5, 0), (0.5, 0.5, 0.5)])
_HH = [
    np.kron(P, P) for P in ([[0, 1], [1, 0]], [[0, -1j], [1j, 0]], [[1, 0], [0, -1]])
]


def c1c2c3(U):
    """Weyl-chamber coordinates of a 4x4 unitary, in units of pi."""
    return np.abs(GateInvariants.from_unitary(np.asarray(U, dtype=complex)).weyl)


def completion(A, B):
    """M(A, B): B on the code space {|01>,|10>}, A on the complement {|00>,|11>}."""
    U = np.zeros((4, 4), dtype=complex)
    U[np.ix_([1, 2], [1, 2])] = B
    U[np.ix_([0, 3], [0, 3])] = A
    return U


def fold(c):
    """Chamber image of canonical parameters c (units of pi)."""
    return c1c2c3(expm(0.5j * np.pi * sum(ci * H for ci, H in zip(c, np.array(_HH)))))


def fit_faces(pts, tol=1e-9):
    """Exact planes a . c = d (small-integer a, rational d) covering all of pts."""
    cand = []
    for a in product((0, 1, -1), repeat=3):
        a = np.array(a)
        if not a.any() or a[np.nonzero(a)[0][0]] < 0:
            continue  # canonical sign only
        v = pts @ a
        for d in {Fraction(x).limit_denominator(16) for x in np.unique(np.round(v, 6))}:
            on = np.abs(v - float(d)) < tol
            if on.sum() < 0.02 * len(pts):
                continue
            if np.linalg.svd(pts[on] - pts[on].mean(0), compute_uv=False)[1] > 0.05:
                cand.append((a, d, on))  # genuinely 2D, not just an edge
    cand.sort(key=lambda c: -c[2].sum())
    faces, covered = [], np.zeros(len(pts), bool)
    for a, d, on in cand:
        if (on & ~covered).any():
            faces.append((a, d))
            covered |= on
    assert covered.all(), "sample farther than tol from every fitted face"
    return faces


def clip_to_chamber(a, d):
    """Ordered polygon of the plane a . c = d inside the chamber tetrahedron."""
    f = CHAMBER @ a - float(d)
    V = [c for c, fc in zip(CHAMBER, f) if abs(fc) < 1e-12]
    for i, j in combinations(range(4), 2):
        if f[i] * f[j] < -1e-24:
            V.append(CHAMBER[i] + f[i] / (f[i] - f[j]) * (CHAMBER[j] - CHAMBER[i]))
    V = np.unique(np.round(V, 12), axis=0)
    b1, b2 = V[0] - V.mean(0), np.cross(a, V[0] - V.mean(0))
    return V[np.argsort(np.arctan2((V - V.mean(0)) @ b2, (V - V.mean(0)) @ b1))]


def family_polyline(B):
    """Fold of the admissible line c1 = c2 = phi_B/pi, c3 in [0, 1]."""
    f = np.arccos(np.clip(abs(B[0, 0]), 0, 1)) / np.pi
    return np.array([fold((f, f, t)) for t in sorted({0.0, f, 0.5, 1 - f, 1.0})])


def seg_dist(p, s0, s1):
    t = np.clip((p - s0) @ (s1 - s0) / max((s1 - s0) @ (s1 - s0), 1e-30), 0, 1)
    return np.linalg.norm(p - (s0 + t * (s1 - s0)))


def retained_unitaries(gate):
    """U(C_r) for every retained realization combination of a 1Q logical gate.

    Translates the gate through the encoding's logical stage, looks up each
    op's realization set, and composes one 4x4 unitary per combination.
    """
    qc = QuantumCircuit(1)
    qc.append(gate, [0])
    per_op = []
    for inst in DualRail.logical_pm().run(qc).data:
        if inst.operation.name == "barrier":
            continue
        lop = getattr(DualRail, inst.operation.name)(inst.operation)
        per_op.append([Operator(r).data for r in lop.realizations])
    out = []
    for combo in product(*per_op):
        U = np.eye(4, dtype=complex)
        for u in combo:
            U = u @ U
        out.append(U)
    return out

In [ ]:
def completion_figure(u_logical, ax, n=1500, seed=7):
    """Draw the three completion tiers of logical gate u_logical on a chamber axis."""
    B = Operator(u_logical).data
    w = WeylChamber()
    w.labels = {}
    w.render(ax)

    # completions tier: exact planar faces fitted to deterministic Haar samples of A
    rng = np.random.default_rng(seed)
    pts = np.array(
        [c1c2c3(completion(random_unitary(2, seed=rng).data, B)) for _ in range(n)]
    )
    polys = [clip_to_chamber(a, d) for a, d in fit_faces(pts)]
    ax.add_collection3d(
        Poly3DCollection(
            polys, facecolor="C0", edgecolor="C0", lw=0.5, alpha=0.5, zorder=1
        )
    )

    # admissible tier: exact fold of the family A = e^{i gamma} Rz(beta)
    line = family_polyline(B)
    ax.plot(*line.T, color="C1", lw=3, solid_capstyle="round", zorder=5)

    # retained tier: U(C_r) of each realization the encoding actually keeps
    retained = [c1c2c3(u) for u in retained_unitaries(u_logical)]
    for r in retained:
        assert min(seg_dist(r, *s) for s in zip(line[:-1], line[1:])) < 1e-9
        ax.scatter(*r, color="red", s=100, zorder=10)
    r = retained[0]

    # depth cues: base-plane footprint of the faces, drop line under the point
    base = np.unique(np.round(np.vstack(polys)[:, :2], 12), axis=0)
    hull = base[ConvexHull(base).vertices]
    ax.add_collection3d(
        Poly3DCollection(
            [np.column_stack([hull, np.zeros(len(hull))])],
            facecolor="gray",
            edgecolor="none",
            alpha=0.2,
            zorder=-10,
        )
    )
    ax.add_collection3d(
        Line3DCollection(
            [[tuple(r), (r[0], r[1], 0)]],
            colors="gray",
            linestyles="--",
            linewidths=0.8,
            alpha=0.4,
            zorder=-10,
        )
    )
    ax.scatter(*r[:2], 0, color="gray", s=15, alpha=0.25, zorder=-10)

In [ ]:
with plt.style.context("lusca"):
    fig = plt.figure(figsize=(6, 4))
    for i, gate in enumerate(TARGETS):
        ax = fig.add_subplot(
            1, len(TARGETS), i + 1, projection="3d", computed_zorder=False
        )
        ax.set_proj_type("persp")
        completion_figure(gate, ax)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig.savefig(
        FIG_DIR / "unitary_completions.pdf",
        bbox_inches="tight",
        metadata={"CreationDate": None},
    )
    plt.show()